# Eel — стандартная библиотека

Всё, что ниже, **встроено в компилятор** — импортов нет. Встроенные имена (`print`, `zip`,
`enumerate`, `range`, `send`, `self`, `spawn`) разрешаются как обычные, поэтому их можно
переопределить своим `def`; методы (`s.split()`, `d.get()`) — whitelist стандартной библиотеки.

Ячейка исполняется на BEAM: Python её не выполняет, `eel` компилирует её в Erlang и гоняет
в persistent-REPL.

## print — печать значений

Строки печатаются как текст, атомы — без кавычек, контейнеры — в питоновском виде, словарь — по порядку ключей.

In [ ]:
print("hi")
print([1, 'a', "s"])
print((1, 2))
print({'b': 2, 'a': 1})
print(2.5)
print(True)
print(None)

hi
[1, a, s]
(1, 2)
{a: 1, b: 2}
2.5
true
nil


`print` возвращает `nil`, поэтому годится внутри выражения:

In [ ]:
print("сначала это")
42

сначала это


42


## range — списки чисел

`range(stop)`, `range(start, stop)`, `range(start, stop, step)` — питоновская семантика, но **неленивая**: возвращается список.

In [ ]:
(range(5), range(2, 5), range(0, 10, 3))

{[0,1,2,3,4],[2,3,4],[0,3,6,9]}


In [ ]:
(range(10, 0, -3), range(5, 5), range(5, 0), range(-3))

{[10,7,4,1],[],[],[]}


## zip — пары до конца короткого списка

In [ ]:
zip([1, 2, 3], ["a", "b"])

[{1,<<"a">>},{2,<<"b">>}]


In [ ]:
zip([], [1])

[]


## enumerate — номер и значение

Второй аргумент задаёт начало нумерации.

In [ ]:
(enumerate(["a", "b"]), enumerate(["a", "b"], 10))

{[{0,<<"a">>},{1,<<"b">>}],[{10,<<"a">>},{11,<<"b">>}]}


## Вместе с включениями

Встроенные работают в шапке генератора; цель-кортеж разбирается образцом.

In [ ]:
[x * 2 for i, x in enumerate([10, 20])]

erl: [X * 2 || {I, X} <- eel_iter(eel_enumerate(0, [10, 20]))]
[20,40]


In [ ]:
[a + b for a, b in zip([1, 2], [10, 20, 30])]

erl: [A + B || {A, B} <- eel_iter(eel_zip([1, 2], [10, 20, 30]))]
[11,22]


In [ ]:
[(i, a * b) for i, (a, b) in enumerate(zip([1, 2], [10, 20]))]

erl: [{I, A * B} || {I, {A, B}} <- eel_iter(eel_enumerate(0, eel_zip([1, 2], [10, 20])))]
[{0,10},{1,40}]


## Методы строк

`startswith`, `endswith`, `lower`, `upper`, `strip`, `lstrip`, `rstrip`, `split`, `join`, `replace`, `splitlines`.

In [ ]:
"  Hello, World  ".strip().lower().split(", ")

[<<"hello">>,<<"world">>]


In [ ]:
("abc".startswith("ab"), "abc".endswith("bc"), "AbC".upper())

{true,true,<<"ABC">>}


In [ ]:
",".join(["a", "b", "c"])

<<"a,b,c">>


In [ ]:
"aaa".replace("a", "b")

<<"bbb">>


In [ ]:
"a\r\nb\nc".splitlines()

[<<"a">>,<<"b">>,<<"c">>]


In [ ]:
(" a  b ".split(), " a  b ".split(" "))

{[<<"a">>,<<"b">>],[<<>>,<<"a">>,<<>>,<<"b">>,<<>>]}


## Методы словаря

`keys`, `values`, `items`, `get` — `get` без ключа даёт `nil`, с аргументом — значение по умолчанию. Порядок ключей в map произволен (в BEAM его нет), поэтому `keys()`/`items()` порядок не гарантируют.

In [ ]:
d = {'a': 1, 'b': 2}
(d.keys(), d.values(), d.items())

{[b,a],[2,1],[{b,2},{a,1}]}


In [ ]:
d.get('a')

1


In [ ]:
(d.get('zzz'), d.get('zzz', 0))

{nil,0}


## count у списков и кортежей — строгое сравнение (как образцы)

In [ ]:
([1, 2, 1].count(1), (7, 7, 7).count(7), [1].count(1.0))

{2,3,0}


## Ok / Error — прелюдия

`Ok(x)` → `{ok, x}`, `Error(e)` → `{error, e}`.

In [ ]:
(Ok(1), Error("boom"))

{{ok,1},{error,<<"boom">>}}


In [ ]:
match Ok(42):
    case Ok(v):
        v
    case Error(e):
        0

42


## Ошибки типов

Не-строка в `join` — не тихая порча, а падение; атом — не строка (одинарные кавычки — атомы).

In [ ]:
",".join([1, 2])

eel runtime error: {error,{bad_argument,join,1}}


In [ ]:
"dggergeg".startswith("dd")

false


In [ ]:
'dggergeg'.startswith('dd')

eel runtime error: {error,{bad_receiver,startswith,dggergeg}}


## Процессы: send / self / spawn / receive

`receive` селективный: сообщение, не подошедшее под образцы, остаётся в ящике; `Timeout(ms)` — ветка `after`.

In [ ]:
def ping():
    parent = self()
    pid = spawn(lambda: send(parent, 'hi'))
    match receive():
        case msg:
            (msg, pid)

In [ ]:
ping()

{hi,<0.84.0>}


In [ ]:
def timed():
    match receive():
        case Ok(v):
            v
        case Timeout(10):
            'timeout'


In [ ]:
timed()

timeout


## Встроенные переопределяемы (по имени, как в Python)

Это последняя ячейка — после неё `zip` в этой сессии уже пользовательский.

In [ ]:
def zip(a, b):
    (a, b)

zip(1, 2)

{1,2}
